# Clasificación con el dataset Iris usando scikit-learn
---
En esta práctica revisaremos:
- Como utilizar Árboles de Decisión (DecisionTree) para clasificar
- Como utilizar Bosques Aleatorios (RandomForest) para clasificar

## 1. Importar bibliotecas a utilizar

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

print("Bibliotecas importadas correctamente")

## 2. Cargar y explorar el dataset

El dataset Iris contiene información del tamaño del sépalo y del pétalo de tres especies de flor Iris.

In [ ]:
iris = load_iris()
X = iris.data # [sepal length, sepal width, petal length, petal width]
y = iris.target #0=setosa, 1=versicolor, 3=virginica

print(f"Forma del dataset : {X.shape} -> {X.shape[0]} muestras, {X.shape[1]} características (atributos)")
print(f"Características   : {iris.feature_names}")
print(f"Clases            : {iris.target_names.tolist()}")


## 3. División del dataset (Train / Test)

Utilizamos el parámetro `stratify=y` para mantener la misma proporción de clases en ambos subconjuntos (Train y Test).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
    )

print(f"Entrenamiento: {X_train.shape[0]} muestras")
print(f"Prueba: {X_test.shape[0]} muestras")
print(f"Clases en entrenamiento: {np.bincount(y_train)}")
print(f"Clases en prueba: {np.bincount(y_test)}")


## 4. Modelo 1 - Decision Tree Classifier

### Hiperparámetros clave
| Parámetro | Valor | Efecto |
|-----------|-------|--------|
| `max_depth` | 4 | Limita la profundidad para evitar overfitting |
| `min_samples_split` | 5 | Nodo se divide solo si tiene ≥5 muestras |
| `random_state` | 42 | Reproducibilidad |

In [ ]:
# Arquitectura del modelo
dt = DecisionTreeClassifier(
    max_depth=4,
    min_samples_split=5,
    random_state = 42
)

# Entrenas el modelo
dt.fit(X_train, y_train)

# Predicciones
y_pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)

print(f"Profundidad real: {dt.get_depth()}")
print(f"Número de hojas : {dt.get_n_leaves()}")
print(f"Exactitud (Accuracy): {acc_dt:.2f}")
print("Reporte de clasificación:")
print(classification_report(y_test, y_pred_dt, target_names=iris.target_names.tolist()))


### Visualización del Árbol de Decisión

In [ ]:
fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(
    dt,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    fontsize=9,
    ax=ax,
    impurity=True,
    rounded=True,
)
ax.set_title("Árbol de Decisión entrenado (max_depth=4)", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

### Importancia de Características — Decision Tree

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
importancias_dt = dt.feature_importances_
indices = np.argsort(importancias_dt)
colores_bar = ["#e74c3c", "#e67e22", "#2ecc71", "#3498db"]

bars = ax.barh(
    [iris.feature_names[i] for i in indices],
    importancias_dt[indices],
    color=[colores_bar[i] for i in indices],
    edgecolor="white"
)
for bar, val in zip(bars, importancias_dt[indices]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=10)

ax.set_xlabel("Importancia (Gini)")
ax.set_title("Importancia de Características — Decision Tree")
ax.set_xlim(0, 0.65)
plt.tight_layout()
plt.show()

## 5. Modelo 2 - Random Forest Classifier

### Hiperparámetros clave
| Parámetro | Valor | Efecto |
|-----------|-------|--------|
| `n_estimators` | 320 | Número de árboles en el bosque |
| `max_depth` | 4 | Profundidad máxima por árbol |
| `max_features` | `"sqrt"` | Características evaluadas por split: √4 = 2 |
| `random_state` | 42 | Reproducibilidad |

In [ ]:
rf = RandomForestClassifier(
    n_estimators=320,
    max_depth=4,
    max_features="sqrt",
    random_state=42
)
rf.fit(X_train,y_train)

y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"Número de árboles:  {rf.n_estimators}")
print(f"Exactitud: {acc_rf:.2f} ({acc_rf*100:.2f}%)\n")
print("Reporte de clasificación:")
print(classification_report(y_test, y_pred_rf, target_names=iris.target_names.tolist()))

### Importancia de Características — Random Forest

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
modelos_imp = [
    (dt.feature_importances_, "Decision Tree", "#e74c3c"),
    (rf.feature_importances_, "Random Forest", "#2ecc71"),
]

for ax, (imps, titulo, color) in zip(axes, modelos_imp):
    indices = np.argsort(imps)
    ax.barh([iris.feature_names[i] for i in indices], imps[indices],
            color=color, alpha=0.85, edgecolor="white")
    ax.set_xlabel("Importancia")
    ax.set_title(titulo)
    ax.set_xlim(0, 0.7)
    for i, val in enumerate(imps[indices]):
        ax.text(val + 0.01, i, f"{val:.3f}", va="center", fontsize=9)

plt.suptitle("Comparación de Importancia de Características", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Validación Cruzada (K-Fold Validation)

La validación cruzada evalúa el modelo en 5 particiones distintas del dataset, dando una estimación más robusta del rendoimiento real del modelo.

In [ ]:
cv_dt = cross_val_score(dt, X, y, cv=5, scoring="accuracy")
cv_rf = cross_val_score(rf, X, y, cv=5, scoring="accuracy")

print(f"Decision Tree - Media: {cv_dt.mean():.4f} +- {cv_dt.std():.4f}")
print(f"\tFolds: {[round(v,4) for v in cv_dt.tolist()]}")
print()
print(f"Random Forest - Media: {cv_rf.mean():.4f} +- {cv_rf.std():.4f}")
print(f"\tFolds: {[round(v,4) for v in cv_rf.tolist()]}")


## 7. Matrices de Confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (y_p, titulo) in zip(axes,[
    (y_pred_dt, "Decision Tree"),
    (y_pred_rf, "Random Forest")]):

   cm = confusion_matrix(y_test, y_p)
   disp = ConfusionMatrixDisplay(cm, display_labels=iris.target_names)
   disp.plot(ax=ax, colorbar=True, cmap="Greens")
   ax.set_title(f"Matriz de Confusión - {titulo}", fontsize=12)

plt.tight_layout()
plt.show()